In [22]:
import os
import json


In [23]:
data_dir = "./data"

session = "CarpeDiem-26-05-2026"

source_dir = os.path.join(
	data_dir,
	"source",
	session
)

raw_dir = os.path.join(
	data_dir,
	"raw",
	session
)



In [24]:
def load_json(path):
	with open(path, encoding="utf-8") as f:
		return json.load(f)

def save_json(path: str, data):
	if not path.endswith(".json"):
		path += ".json"
	
	with open(path, "w", encoding="utf-8") as f:
		json.dump(data, f, ensure_ascii=False, indent=4)


In [25]:
def normalize_token(token: str):
	if token.startswith("18_"):
		return token[3:]

	return token


In [26]:
def strip_strings(data: dict):
    for response in data["responses"]:
        for key in list(response):
            if isinstance(key, str):
                value = response.pop(key)
                response[key.strip()] = value

    return data


In [27]:
for phase in ["Pre", "Post"]:
	phase_source_dir = os.path.join(source_dir, phase)
	phase_raw_dir = os.path.join(raw_dir, phase)
	os.makedirs(phase_raw_dir, exist_ok=True)

	code_file = os.path.join(phase_source_dir, "code.json")
	full_file = os.path.join(phase_source_dir, "full.json")

	raw_code_file = os.path.join(phase_raw_dir, "code.json")
	raw_full_file = os.path.join(phase_raw_dir, "full.json")

	code_data = load_json(code_file)
	full_data = strip_strings(load_json(full_file))

	print(full_data)
	
	full_by_token = {
		normalize_token(response["Código de acceso"]): response
		for response in full_data.get("responses", [])
		if response.get("Código de acceso")
	}

	code_result = {}
	full_result = {}

	for code_response in code_data.get("responses", []):
		original_token = code_response.get("token")

		if not original_token:
			print(f"{phase}: respuesta sin token")
			continue

		token = normalize_token(original_token)

		full_response = full_by_token.get(token)

		if full_response is None:
			print(f"{phase}: token '{token}' no encontrado en full")
			continue

		code_response["token"] = token
		full_response["Código de acceso"] = token

		code_result[token] = code_response
		full_result[token] = full_response

	save_json(raw_code_file, code_result)
	save_json(raw_full_file, full_result)

	print(f"{session}/{phase}: {len(code_result)} respuestas")


{'responses': [{'ID de respuesta': 1, 'Fecha de envío': None, 'Última página': -1, 'Lenguaje inicial': 'es', 'Semilla': '789321124', 'Código de acceso': 'memogames', 'Fecha de inicio': '2026-05-25 19:59:01', 'Fecha de la última acción': '2026-05-25 19:59:15', 'Edad': None, 'Género': '', 'Género [Otro]': '', 'Curso': '', 'Curso [Otro]': '', '¿Cómo de peligrosas consideras las siguientes acciones? [Tener tu perfil en público]': '', '¿Cómo de peligrosas consideras las siguientes acciones? [Agregar a contactos o aceptar como seguidores a personas que no conoces en la vida real]': '', '¿Cómo de peligrosas consideras las siguientes acciones? [Unirte a grupos o comunidades o servidores públicos]': '', '¿Cómo de peligrosas consideras las siguientes acciones? [Hablar por chat (escrito o de voz) con alguien que no conoces en la vida real]': '', '¿Cómo de peligrosas consideras las siguientes acciones? [Hablar por videollamada con alguien que no conoces en la vida real]': '', '¿Cómo de peligrosas 